In [ ]:
"""
This notebook contains the instructions to run the complete pySCENIC workflow to estimate TFs based on your gene expression data.

Important stuff to remember:
1. Make sure you have created a conda environment. Many of the dependencies required for pySCENIC can conflict with other packages you may have installed. Creating a separate conda environment will help avoid these conflicts.
2. Make sure you have created a .loom file from your gene expression data. See the conversions script in this repository for instructions on how to do this.
3. Make sure you have downloaded the required databases for pySCENIC. See https://resources.aertslab.org/cistarget/tf_lists/ for the TFs list based on species of interest
and https://resources.aertslab.org/cistarget/ for the motif and ranking databases required for pySCENIC. Downloading the ranking databases and motifs is optinal, since we will execute that in the code below.
4. Adjust the number of workers based on the number of CPU cores available on your machine. More workers will speed up processing but will also use more memory.

"""

In [ ]:
import os

# Create directory for databases
os.makedirs('databases', exist_ok=True)

In [ ]:
#obtain both 10kbp and 500bp mouse mm10 feather databases and the motif to TF mapping file
!wget https://resources.aertslab.org/cistarget/databases/mus_musculus/mm10/refseq_r80/mc_v10_clust/gene_based/mm10_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather -O databases/mm10_10kbp.feather
!wget https://resources.aertslab.org/cistarget/databases/mus_musculus/mm10/refseq_r80/mc_v10_clust/gene_based/mm10_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather -O databases/mm10_500bp.feather
!wget https://resources.aertslab.org/cistarget/motif2tf/motifs-v10nr_clust-nr.mgi-m0.001-o0.0.tbl -O databases/motifs.tbl

--2025-11-18 20:42:02--  https://resources.aertslab.org/cistarget/databases/mus_musculus/mm10/refseq_r80/mc_v10_clust/gene_based/mm10_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather
Resolving resources.aertslab.org (resources.aertslab.org)... 134.58.50.9
Connecting to resources.aertslab.org (resources.aertslab.org)|134.58.50.9|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 237172098 (226M)
Saving to: ‘databases/mm10_10kbp.feather’

databases/mm10_10kb 100%[===================>] 226.18M   106MB/s    in 2.1s    

2025-11-18 20:42:05 (106 MB/s) - ‘databases/mm10_10kbp.feather’ saved [237172098/237172098]

--2025-11-18 20:42:05--  https://resources.aertslab.org/cistarget/databases/mus_musculus/mm10/refseq_r80/mc_v10_clust/gene_based/mm10_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather
Resolving resources.aertslab.org (resources.aertslab.org)... 134.58.50.9
Connecting to resources.aertslab.org (resources.aertslab.org)|

In [ ]:
#install scanpy for loom file handling. You must also have downloaded the TF txt file from https://resources.aertslab.org/cistarget/tf_lists/
!pip install scanpy
import scanpy as sc

f_tfs = "/kaggle/input/mouse-tfs/allTFs_mm.txt"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 34.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.3/172.3 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 90.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 97.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.2/58.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.4/276.4 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 60.0 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 4.4 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.2.2
    Uninstalling scikit-learn-1.2.2:
      Successfully uninstalled scikit-learn-1.2.2
  Attempting uninstall: matplotlib
    Found existing installation: matplotlib 3

In [ ]:
#Install pyscenic. You may wish to uncommoment the second line should you choose to install directly from the github. 
#Note that the github version may be more up to date but could also be less stable.
!pip install pyscenic
#!pip install git+https://github.com/aertslab/pySCENIC.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.3/53.3 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 76.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 19.6 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 47.3 MB/s eta 0:00:00
  Created wheel for interlap: filename=interlap-0.2.7-py3-none-any.whl size=6282 sha256=e5d642c2b92de761606c459f0aa993f15e097d30a57bc5c55dee1e405a86a6c8
  Stored in directory: /root/.cache/pip/wheels/dd/c0/23/cb81b8babdc3a0b751c69ebf227f83b0e1640bef7ffedf0386
  Created wheel for loompy: filename=loompy-3

In [ ]:
#Set file paths and import libraries
import os
import pyscenic
import pandas as pd
import loompy as lp

# Define file paths
loom_file = "/kaggle/input/gallitano-loom/gallitano loom.loom"  # Your input loom file
adjacencies_file = "adjacencies.csv"
regulons_file = "regulons.csv"
auc_mtx_file = "auc_matrix.csv"

# Database files (adjust for human/mouse)
db_fnames = [
    'databases/mm10_10kbp.feather',
    'databases/mm10_500bp.feather'
]
motif_annotations = 'databases/motifs.tbl'

In [ ]:
#Fix loom file structure from R conversion
import numpy as np

# Read your existing loom
with lp.connect('/kaggle/input/gallitano-loom-final/gallitano_loom.loom', mode='r', validate=False) as ds_in:
    # Extract all data
    matrix = ds_in[:, :]
    row_attrs = {key: ds_in.ra[key][:] for key in ds_in.ra.keys()}
    col_attrs = {key: ds_in.ca[key][:] for key in ds_in.ca.keys()}
    
print(f"Read {matrix.shape[0]} genes x {matrix.shape[1]} cells")

# Create new loom with proper structure
lp.create(
    'gallitano_fixed.loom',
    matrix,
    row_attrs,
    col_attrs
)

print("Fixed loom created: gallitano_fixed.loom")

Read 23013 genes x 23097 cells
Fixed loom created: gallitano_fixed.loom


In [ ]:
#pySCENIC has a number of compatability issues with recent versions of dask, distributed, and numpy. 
# To avoid these issues, please install the following specific versions of these packages:
!pip install dask-expr==0.5.3 distributed==2024.2.1
!pip install 'numpy>=1.23,<1.24'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.6/171.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 44.0 MB/s eta 0:00:00
  Attempting uninstall: dask
    Found existing installation: dask 2024.12.1
    Uninstalling dask-2024.12.1:
      Successfully uninstalled dask-2024.12.1
  Attempting uninstall: distributed
    Found existing installation: distributed 2024.12.1
    Uninstalling distributed-2024.12.1:
      Successfully uninstalled distributed-2024.12.1
  Attempting uninstall: dask-expr
    Found existing installation: dask-expr 1.1.21
    Uninstalling dask-expr-1.1.21:
      Successfully uninstalled dask-expr-1.1.21
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rapids-dask-dependency 25.2.0 requires dask==2024.12.1, but you have dask 2024.2.1 w

In [ ]:
#Rename the feather database files to match expected pattern for pyscenic

# Check what you have
!ls -lh databases/

# Rename to match expected pattern
!mv databases/mm10_10kbp.feather \
    databases/mm10_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather

!mv databases/mm10_500bp.feather \
    databases/mm10_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather

total 562M
-rw-r--r-- 1 root root 227M Jul  8  2022 mm10_10kbp.feather
-rw-r--r-- 1 root root 228M Jul  8  2022 mm10_500bp.feather
-rw-r--r-- 1 root root 108M Aug 17  2022 motifs.tbl


In [ ]:
#Execute the gene regulatory network inference step to generate the adjacency matrix
#adjust num_workers to number of available CPU cores. More cores = faster processing but more memory usage.
!pyscenic grn gallitano_fixed.loom /kaggle/input/mouse-tfs/allTFs_mm.txt \
    -o adj.csv \
    --num_workers 4 \ 
    --seed 123 


2025-11-18 20:45:19,591 - pyscenic.cli.pyscenic - INFO - Loading expression matrix.

2025-11-18 20:45:31,523 - pyscenic.cli.pyscenic - INFO - Inferring regulatory networks.
preparing dask client
parsing input
creating dask graph
4 partitions
computing dask graph
/usr/local/lib/python3.11/dist-packages/distributed/client.py:3169: UserWarning: Sending large graph of size 3.94 GiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(
not shutting down client, client was created externally
finished

2025-11-19 03:33:35,010 - pyscenic.cli.pyscenic - INFO - Writing results to file.


In [ ]:
#Sanity check of output adjacency file
import os

# Check output file
if os.path.exists('adj.csv'):
    size_mb = os.path.getsize('adj.csv') / (1024*1024)
    print(f"adj.csv size: {size_mb:.2f} MB")
    
    # Count lines (rough progress indicator)
    with open('adj.csv', 'r') as f:
        lines = sum(1 for line in f)
    print(f"Lines in output: {lines:,}")
    print(f"This represents ~{lines} adjacencies so far")
else:
    print("No output file yet - still in setup phase")

adj.csv size: 62.66 MB
Lines in output: 1,977,294
This represents ~1977294 adjacencies so far


In [ ]:
#Second sanity check to ensure the adjacency file looks correct
adjacencies.head()

,"TF,target,importance"
0,"Rps10,Rpl35a,21.742859834578386"
1,"Rps10,Rpl27a,21.108064445817927"
2,"Rps10,Rplp2,20.950330921931968"
3,"Prnp,Tspan7,18.476635975225456"
4,"Rps10,Rpl11,18.372365030560434"


In [ ]:
#Run the cisTarget step to identify regulons from the adjacency matrix
!pyscenic ctx \
    adj.csv \
    databases/mm10_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather \
    databases/mm10_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather \
    --annotations_fname databases/motifs.tbl \
    --expression_mtx_fname gallitano_fixed.loom \
    --output regulons.csv \
    --num_workers 4 #adjust this based on the number of available CPU cores. More cores = faster processing but more memory usage.

In [ ]:
#Adjust pandas version to avoid compatability issues with AUC step. 
#Generating a .loom file as output requires you to execute this step. 
!pip install "pandas<2.0"

In [ ]:
#Calculate the regulon activity scores (AUC) for each cell
#you can also change the extension of the output file to .csv if you wish
!pyscenic aucell \
    gallitano_fixed.loom \
    regulons.csv \
    --output auc_mtx.loom \
    --num_workers 1 #Adjust based on available CPU cores.


2025-11-19 06:02:09,431 - pyscenic.cli.pyscenic - INFO - Loading expression matrix.

2025-11-19 06:02:22,932 - pyscenic.cli.pyscenic - INFO - Loading gene signatures.
Create regulons from a dataframe of enriched features.
Additional columns saved: []

2025-11-19 06:02:26,154 - pyscenic.cli.pyscenic - INFO - Calculating cellular enrichment.
100%|█████████████████████████████████████████| 130/130 [00:40<00:00,  3.22it/s]

2025-11-19 06:05:23,497 - pyscenic.cli.pyscenic - INFO - Writing results to file.


In [ ]:
#This step only applies if you generated a .csv file as output for the AUC step.

auc = pd.read_csv("auc_mtx.csv", index_col=0)  # rows = regulons, cols = cells
print(auc.shape)
print(auc.head())

(23097, 130)
                      Alx4(+)   Atf2(+)  Atf5(+)  Bcl11a(+)  Bclaf1(+)  \
Cell                                                                     
AAACCCAAGAAGCGCT-9   0.024889  0.030032      0.0   0.055985   0.059179   
AAACCCAAGCGTGTCC-47  0.005571  0.002987      0.0   0.009981   0.030793   
AAACCCAAGGACGGAG-34  0.002692  0.002360      0.0   0.009622   0.027436   
AAACCCAAGGGCAACT-2   0.001261  0.008773      0.0   0.024730   0.028200   
AAACCCAAGGTTCTTG-47  0.000000  0.020923      0.0   0.005955   0.030965   

                      Bptf(+)  Cebpa(+)  Cebpb(+)  Cebpd(+)  Cebpg(+)  ...  \
Cell                                                                   ...   
AAACCCAAGAAGCGCT-9   0.052016  0.025359  0.010265  0.016904  0.034509  ...   
AAACCCAAGCGTGTCC-47  0.039431  0.034373  0.024806  0.054217  0.046709  ...   
AAACCCAAGGACGGAG-34  0.032761  0.037696  0.023563  0.049781  0.048410  ...   
AAACCCAAGGGCAACT-2   0.025876  0.145234  0.112373  0.088230  0.091888  ...   


In [ ]:
#Congrats on completing the pySCENIC workflow. You can use the .csv or .loom files generated for downstream analysis and visualization.
#See the other scripts in this repository for examples of downstream analysis and visualization.